# GMRES — Interactive Experiments

Compare unrestarted and restarted GMRES, optional Jacobi/ILU preconditioning, and SciPy GMRES.

Select settings and click **Run experiment**. All matrices use a fixed random seed for reproducibility.


In [2]:
from pathlib import Path
import sys
import time

# Discover the checkout root in local Jupyter and Binder sessions.
for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / "krylov_methods").is_dir():
        root = candidate
        break
else:
    raise RuntimeError("Launch Jupyter from inside the repository checkout.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from krylov_methods.notebooks.notebook_utils import (
    spd_problem, nonsymmetric_problem, residual_plot, error_plot, report,
)

from scipy.sparse.linalg import gmres as scipy_gmres
from krylov_methods.gmres_solver.GMRES import gmres

from scipy.sparse.linalg import LinearOperator
from krylov_methods.preconditioners import make_preconditioner


## Interactive experiment

The curves show the *true* residual $\|b-Ax_k\|_2$. The dashed line is the absolute/relative stopping threshold.

In [4]:
system = widgets.Dropdown(
    options=[("Nonsymmetric", "nonsymmetric"), ("Real SPD", "spd")],
    description="System",
)
size = widgets.IntSlider(value=30, min=5, max=100, step=5, description="Size n")
restart = widgets.IntSlider(value=12, min=2, max=40, step=2, description="Restart m")
condition = widgets.FloatLogSlider(value=1e4, base=10, min=1, max=6, step=1, description="Cond(A), SPD")
tol = widgets.FloatLogSlider(value=1e-9, base=10, min=-12, max=-5, step=1, description="rtol")
budget = widgets.IntSlider(value=200, min=20, max=600, step=20, description="Max iterations")
precond = widgets.Dropdown(
    options=["none", "jacobi", "ilu", "lu", "ssor", "cholesky", "ic0", "custom (diagonal)"],
    description="Precond",
)
button = widgets.Button(description="Run experiment", button_style="success")
output = widgets.Output()


def run(_):
    with output:
        clear_output(wait=True)
        A, b, x_true = (
            nonsymmetric_problem(size.value)
            if system.value == "nonsymmetric"
            else spd_problem(size.value, condition.value)
        )
        if system.value != "spd" and precond.value in ("cholesky", "ic0"):
            print("Cholesky and IC(0) require an SPD system; select 'Real SPD'.")
            return

        m = min(restart.value, size.value)
        chosen = precond.value
        custom = chosen == "custom (diagonal)"
        p = "custom" if custom else None if chosen == "none" else chosen
        M = (lambda v: v / np.diag(A)) if custom else None
        threshold = max(1e-12, tol.value * np.linalg.norm(b))
        eigs = np.linalg.eigvals(A)
        print("SYSTEM INFORMATION")
        print(f"Matrix: {system.value}; dimension={size.value}; dense; dtype={A.dtype}")
        print(f"Symmetry/Hermitian error ||A-Aᴴ||₂={np.linalg.norm(A-A.conj().T, ord=2):.3e}")
        print(f"Condition number κ₂(A)={np.linalg.cond(A):.3e}")
        print(f"Eigenvalue real parts: [{eigs.real.min():.3e}, {eigs.real.max():.3e}]")
        print(f"Eigenvalue imaginary parts: [{eigs.imag.min():.3e}, {eigs.imag.max():.3e}]")
        print(f"||b||₂={np.linalg.norm(b):.3e}; ||x*||₂={np.linalg.norm(x_true):.3e}")
        print(f"Initial guess x₀=0; initial residual={np.linalg.norm(b):.3e}")
        print(f"True-residual stopping threshold={threshold:.3e}")
        print(f"Restart dimension={m}; total Arnoldi budget={budget.value}")
        print(f"Preconditioner={chosen}; GMRES uses LEFT preconditioning.\n")

        histories = {}
        estimated_histories = {}
        for label, dim in (("Unrestarted", None), (f"Restarted (m={m})", m)):
            indices = [0]
            true_residuals = [np.linalg.norm(b)]
            estimated_indices = []
            estimated_residuals = []

            def iteration_callback(k, true_residual, estimated_preconditioned_residual):
                indices.append(k)
                true_residuals.append(true_residual)
                estimated_indices.append(k)
                estimated_residuals.append(estimated_preconditioned_residual)

            start = time.perf_counter()
            result = gmres(
                A, b, restart=dim, maxiter=budget.value,
                rtol=tol.value, atol=1e-12, preconditioner=p, M=M,
                callback=iteration_callback,
            )
            elapsed = time.perf_counter() - start
            report("Our " + label, result.x, x_true, b, A,
                   result.iterations, result.converged)
            print(f"  Cycles={result.cycles}; solver-call time={elapsed:.6f} s; "
                  f"status={result.message}")
            # Callback values are true residual norms of intermediate candidate iterates.
            histories[label] = (np.asarray(indices), np.asarray(true_residuals))
            estimated_histories[label] = (
                np.asarray(estimated_indices), np.asarray(estimated_residuals)
            )

        # For a fair comparison, use the SAME left inverse action in SciPy.
        apply_M = make_preconditioner(A, p, M=M)
        scipy_M = LinearOperator(A.shape, matvec=apply_M, dtype=A.dtype)
        scipy_cycles = [0]
        scipy_residuals = [np.linalg.norm(b)]

        def scipy_callback_x(xk):
            scipy_cycles.append(len(scipy_cycles))
            scipy_residuals.append(np.linalg.norm(b - A @ xk))

        # With callback_type='x', SciPy maxiter counts completed cycles.
        # A separate 'pr_norm' pass counts the actual inner iterations,
        # including any shortened final cycle.
        scipy_max_cycles = max(1, budget.value // m)
        start = time.perf_counter()
        scipy_x, info = scipy_gmres(
            A, b, M=scipy_M, restart=m, maxiter=scipy_max_cycles,
            rtol=tol.value, atol=1e-12,
            callback=scipy_callback_x, callback_type="x",
        )
        scipy_time = time.perf_counter() - start
        scipy_inner_residual_estimates = []
        def scipy_callback_pr(value):
            scipy_inner_residual_estimates.append(float(value))
        _, info_count = scipy_gmres(
            A, b, M=scipy_M, restart=m, maxiter=scipy_max_cycles,
            rtol=tol.value, atol=1e-12,
            callback=scipy_callback_pr, callback_type="pr_norm",
        )
        total_inner = len(scipy_inner_residual_estimates)
        if info != info_count:
            print("Warning: SciPy diagnostic runs returned different info values.")

        # A cycle's final callback can occur before m iterations when it
        # converges early. The per-inner-iteration pass identifies that point.
        scipy_iteration_indices = [0] + [
            min(j * m, total_inner) for j in range(1, len(scipy_cycles))
        ]
        if len(scipy_iteration_indices) > 1:
            scipy_iteration_indices[-1] = total_inner
        # Include the explicitly computed final residual, even if the
        # last callback was omitted by SciPy on an early termination.
        final_scipy_residual = np.linalg.norm(b - A @ scipy_x)
        if not np.isclose(scipy_residuals[-1], final_scipy_residual,
                          rtol=1e-12, atol=1e-14):
            if scipy_iteration_indices[-1] == total_inner:
                scipy_residuals[-1] = final_scipy_residual
            else:
                scipy_iteration_indices.append(total_inner)
                scipy_residuals.append(final_scipy_residual)

        report("SciPy restarted", scipy_x, x_true, b, A, total_inner, info == 0)
        print(f"  Restart cycles={len(scipy_cycles)-1}; inner iterations={total_inner}; "
              f"time={scipy_time:.6f} s; info={info}")
        print("  SciPy uses the same selected LEFT preconditioner.\n")
        histories["SciPy restarted"] = (
            np.asarray(scipy_iteration_indices), np.asarray(scipy_residuals)
        )

        residual_plot(
            histories, threshold,
            title="GMRES — True residual by Arnoldi iteration",
            xlabel="Total Arnoldi iterations",
        )

        # Givens estimates are LEFT-preconditioned residual norms and need
        # a separate axis/title from the true residual norms.
        fig, ax = plt.subplots(figsize=(10, 5))
        for label, (indices, values) in estimated_histories.items():
            ax.semilogy(indices, np.maximum(values, np.finfo(float).tiny),
                        marker="o", markersize=3, label=label)
        ax.set(xlabel="Total Arnoldi iterations",
               ylabel=r"Estimated $\|M^{-1}(b-Ax_k)\|_2$",
               title="GMRES — Preconditioned residual estimates")
        ax.grid(True, which="both", alpha=0.25)
        ax.legend()
        fig.tight_layout()
        plt.show()

        print("Interpretation: left-GMRES minimizes the preconditioned residual, "
              "but convergence is verified using the true residual. "
              "Smaller restart dimensions reduce storage but may slow convergence.")


button.on_click(run)
display(widgets.VBox([
    system, size, restart, condition, tol, budget, precond, button, output,
]))


### Notes

Small residual does not always imply small solution error for ill-conditioned systems. Compare both metrics; a preconditioner changes convergence but should not change the exact solution.


## Numerical exploration — Restarting and residual norms

**Restart study.** Smaller Krylov dimensions reduce basis storage but can
slow convergence or cause stagnation. The plot below uses the same system,
preconditioner, stopping criterion, and total Arnoldi budget for each choice.

**Left-preconditioning note.** GMRES reduces the preconditioned residual
$\|M^{-1}(b-Ax)\|_2$, while convergence is accepted only when the **true**
residual $\|b-Ax\|_2$ meets the tolerance. Scaling an inverse action can
change the preconditioned norm dramatically without changing the solution.


In [7]:
restart_study_button = widgets.Button(description="Explore restart dimensions", button_style="info")
restart_study_output = widgets.Output()

def run_restart_study(_):
    with restart_study_output:
        clear_output(wait=True)
        A, b, x_true = (
            nonsymmetric_problem(size.value) if system.value == "nonsymmetric"
            else spd_problem(size.value, condition.value)
        )
        if system.value != "spd" and precond.value in ("cholesky", "ic0"):
            print("Select an SPD system for Cholesky or IC(0).")
            return
        custom = precond.value == "custom (diagonal)"
        kind = "custom" if custom else None if precond.value == "none" else precond.value
        M = (lambda v: v / np.diag(A)) if custom else None
        curves = {}
        for dim in (5, 10, 20, None):
            indices, norms = [0], [np.linalg.norm(b)]
            def cb(k, true_residual, estimated_residual):
                indices.append(k)
                norms.append(true_residual)
            result = gmres(A, b, restart=dim, maxiter=budget.value,
                           rtol=tol.value, atol=1e-12,
                           preconditioner=kind, M=M, callback=cb)
            label = f"GMRES({min(dim, size.value)})" if dim is not None else "Unrestarted"
            curves[label] = (indices, norms)
            report(label, result.x, x_true, b, A, result.iterations, result.converged)
        residual_plot(curves, max(1e-12, tol.value * np.linalg.norm(b)),
                      "Effect of GMRES restart dimension", xlabel="Arnoldi iterations")

restart_study_button.on_click(run_restart_study)
display(widgets.VBox([restart_study_button, restart_study_output]))


In [8]:
scale = widgets.FloatLogSlider(value=1.0, base=10, min=-8, max=8,
                               step=1, description="Scale M⁻¹")
scale_button = widgets.Button(description="Explore residual scaling", button_style="info")
scale_output = widgets.Output()

def run_scale(_):
    with scale_output:
        clear_output(wait=True)
        A = np.diag([1.0, 1e-6])
        x_true = np.array([1.0, 1.0])
        b = A @ x_true
        x0 = np.zeros_like(b)
        true_r = b - A @ x0
        inverse_action = lambda r: scale.value * r
        left_r = inverse_action(true_r)
        print("Illustration: A=diag(1, 10⁻⁶), x₀=0, x*=ones(2)")
        print(f"||b-Ax₀||₂ = {np.linalg.norm(true_r):.3e}")
        print(f"||M⁻¹(b-Ax₀)||₂ = {np.linalg.norm(left_r):.3e}")
        print("The apparent preconditioned norm changes with scaling; "
              "the true residual and exact solution do not.")
        fig, ax = plt.subplots(figsize=(7, 3.5))
        ax.bar(["True residual", "Preconditioned residual"],
               [np.linalg.norm(true_r), max(np.linalg.norm(left_r), np.finfo(float).tiny)])
        ax.set_yscale("log")
        ax.set_ylabel("Norm (log scale)")
        ax.set_title("True vs left-preconditioned residual")
        fig.tight_layout()
        plt.show()

scale_button.on_click(run_scale)
display(widgets.VBox([scale, scale_button, scale_output]))
